### Transformer Implementation using Pytorch

In [1]:
# Imports 
import torch
import torch.nn as nn
from torchinfo import summary

In [2]:
# Parameters
SEQ_LENGTH = 200
VOCAB_SIZE_SRC = 100
VOCAB_SIZE_DST = 120

BATCH_SIZE = 1
D_MODEL = 512
NUM_HEADS = 8 
HEAD_DIM = D_MODEL // NUM_HEADS
HIDDEN_DIM = 2048
N = 6
DROP_PROB = 0.1 

In [3]:
# Embeddings 
class InputEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=VOCAB_SIZE_SRC, embedding_dim = D_MODEL)

    def forward(self, x):
        print(f"original: {x.shape}")

        x = self.embedding(x)
        print(f"after embedding: {x.shape}")

        return x
        
class OutputEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=VOCAB_SIZE_DST, embedding_dim=D_MODEL)

    def forward(self, x):
        print(f"orginalt: {x.shape}")

        x = self.embedding(x)
        print(f'after embeddingt: {x.shape}')

        return x

In [4]:
# Initialize Input Embedding
input_embedding = InputEmbedding()

x_src = torch.randint(0, VOCAB_SIZE_SRC, (BATCH_SIZE, SEQ_LENGTH))
x_src = input_embedding(x_src)

original: torch.Size([1, 200])
after embedding: torch.Size([1, 200, 512])


In [5]:
# Initialize Output Embedding
output_embedding = OutputEmbedding()

x_dst = torch.randint(0, VOCAB_SIZE_DST, (BATCH_SIZE, SEQ_LENGTH))
x_dst = output_embedding(x_dst)

orginalt: torch.Size([1, 200])
after embeddingt: torch.Size([1, 200, 512])


In [6]:
# Positional Encoder
class PositionalEncoding(nn.Module):
    def forward(self):
        pos = torch.arange(SEQ_LENGTH).reshape(SEQ_LENGTH, 1)
        print(f'postt: {pos.shape}')
        
        i = torch.arange(0, D_MODEL, 2) 
        denominator = torch.pow(10000, i/D_MODEL)
        print(f'denominatort: {denominator.shape}')

        even_pos_embed = torch.sin(pos/denominator)
        odd_pos_embed = torch.cos(pos/denominator)
        print(f'even_pos_embed: {even_pos_embed.shape}')

        stacked = torch.stack([even_pos_embed, odd_pos_embed], dim=2)
        print(f'stackedtt: {stacked.shape}')

        pos_embed = torch.flatten(stacked, start_dim=1, end_dim=2)
        print(f'pos_embedt: {pos_embed.shape}')

        return pos_embed

In [7]:
positional_encoding = PositionalEncoding()

positional_embedding = positional_encoding()

postt: torch.Size([200, 1])
denominatort: torch.Size([256])
even_pos_embed: torch.Size([200, 256])
stackedtt: torch.Size([200, 256, 2])
pos_embedt: torch.Size([200, 512])


In [8]:
# Scaled Dot-Product Attention

class Attention(nn.Module):

    def create_mask(self):
        mask = torch.tril(torch.ones((SEQ_LENGTH, SEQ_LENGTH)))
        mask[mask == 0] = -float('inf')
        mask[mask == 1] = 0
        return mask.clone().detach()
    
    def forward(self, q, k, v, look_ahead_mask = False):
        print(f'qttt: {q.shape}')
        print(f'kttt: {k.shape}')
        print(f'vttt: {v.shape}')

        multiplied = torch.matmul(q, k.transpose(-1, -2))
        print(f'multipliedtt: {multiplied.shape}')

        scaled = multiplied / torch.sqrt(torch.tensor(HEAD_DIM))
        print(f'scaledttt: {scaled.shape}')

        if look_ahead_mask == True:
            mask = self.create_mask()
            print(f'maskttt: {mask.shape}')
            scaled += mask 

        attn_output_weights = torch.softmax(scaled, dim=-1)
        print(f'attn_output_weights: {attn_output_weights.shape}')

        attn_output = torch.matmul(attn_output_weights, v)
        print(f'attn_outputtt: {attn_output.shape}')

        return attn_output, attn_output_weights 

In [9]:
# Look-Ahead Mask
attention = Attention()

q = torch.randn(BATCH_SIZE, SEQ_LENGTH, HEAD_DIM)
k = torch.randn(BATCH_SIZE, SEQ_LENGTH, HEAD_DIM)
v = torch.randn(BATCH_SIZE, SEQ_LENGTH, HEAD_DIM)

attn_output, attn_output_weights  = attention(q, k, v, look_ahead_mask=True)

qttt: torch.Size([1, 200, 64])
kttt: torch.Size([1, 200, 64])
vttt: torch.Size([1, 200, 64])
multipliedtt: torch.Size([1, 200, 200])
scaledttt: torch.Size([1, 200, 200])
maskttt: torch.Size([200, 200])
attn_output_weights: torch.Size([1, 200, 200])
attn_outputtt: torch.Size([1, 200, 64])


In [13]:
# Multihead Self-Attention
class SelfAttention(nn.Module):
    def __init__(self, look_ahead_mask=False):
        super().__init__()
        self.look_ahead_mask = look_ahead_mask 

        self.qkv_linear = nn.Linear(D_MODEL, 3*D_MODEL)
        self.attention = Attention()
        self.linear = nn.Linear(D_MODEL, D_MODEL)
    def forward(self, x):
        print(f"originaltt: {x.shape}")

        x = self.qkv_linear(x)  #(1)
        print(f"after qkv_lineart: {x.shape}")

        x = x.reshape(BATCH_SIZE, SEQ_LENGTH, NUM_HEADS, 3*HEAD_DIM)  #(2)
        print(f"after reshapett: {x.shape}")

        x = x.permute(0, 2, 1, 3)  #(3)
        print(f"after permutett: {x.shape}")

        q, k, v = x.chunk(3, dim=-1)  #(4)
        print(f"qttt: {q.shape}")
        print(f"kttt: {k.shape}")
        print(f"vttt: {v.shape}")

        attn_output, attn_output_weights = self.attention(q, k, v, 
                                                          look_ahead_mask=self.look_ahead_mask) #(5)
        print(f"attn_outputtt: {attn_output.shape}")
        print(f"attn_output_weightst: {attn_output_weights.shape}")

        x = attn_output.permute(0, 2, 1, 3)  #(6)
        print(f"after permutett: {x.shape}")

        x = x.reshape(BATCH_SIZE, SEQ_LENGTH, NUM_HEADS*HEAD_DIM)  #(7)
        print(f"after reshapett: {x.shape}")

        x = self.linear(x)  #(8)
        print(f"after lineartt: {x.shape}")

        return x

In [14]:
self_attention = SelfAttention()

x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
x = self_attention(x)

originaltt: torch.Size([1, 200, 512])
after qkv_lineart: torch.Size([1, 200, 1536])
after reshapett: torch.Size([1, 200, 8, 192])
after permutett: torch.Size([1, 8, 200, 192])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
multipliedtt: torch.Size([1, 8, 200, 200])
scaledttt: torch.Size([1, 8, 200, 200])
attn_output_weights: torch.Size([1, 8, 200, 200])
attn_outputtt: torch.Size([1, 8, 200, 64])
attn_outputtt: torch.Size([1, 8, 200, 64])
attn_output_weightst: torch.Size([1, 8, 200, 200])
after permutett: torch.Size([1, 200, 8, 64])
after reshapett: torch.Size([1, 200, 512])
after lineartt: torch.Size([1, 200, 512])


In [17]:
class CrossAttention(nn.Module):

    def __init__(self):
        super().__init__()

        self.kv_linear = nn.Linear(D_MODEL, 2*D_MODEL)  #(1)
        self.q_linear = nn.Linear(D_MODEL, D_MODEL)  #(2)
        self.attention = Attention()
        self.linear = nn.Linear(D_MODEL, D_MODEL)  #(3)

    def forward(self, x_enc, x_dec):  #(1)
        print(f"x_enc originaltt: {x_enc.shape}")
        print(f"x_dec originaltt: {x_dec.shape}")

        x_enc = self.kv_linear(x_enc)  #(2)
        print(f"nafter kv_lineartt: {x_enc.shape}")

        x_enc = x_enc.reshape(BATCH_SIZE, SEQ_LENGTH, NUM_HEADS, 2*HEAD_DIM)  #(3)
        print(f"after reshapett: {x_enc.shape}")

        x_enc = x_enc.permute(0, 2, 1, 3)  #(4)
        print(f"after permutett: {x_enc.shape}")

        k, v = x_enc.chunk(2, dim=-1)  #(5)
        print(f"kttt: {k.shape}")
        print(f"vttt: {v.shape}")

        x_dec = self.q_linear(x_dec)  #(6)
        print(f"nafter q_lineartt: {x_dec.shape}")

        x_dec = x_dec.reshape(BATCH_SIZE, SEQ_LENGTH, NUM_HEADS, HEAD_DIM)  #(7)
        print(f"after reshapett: {x_dec.shape}")

        q = x_dec.permute(0, 2, 1, 3)  #(8)
        print(f"after permute (q)t: {q.shape}")

        attn_output, attn_output_weights = self.attention(q, k, v) #(9)
        print(f"nattn_outputtt: {attn_output.shape}")
        print(f"attn_output_weightst: {attn_output_weights.shape}")

        x = attn_output.permute(0, 2, 1, 3)
        print(f"after permutett: {x.shape}")

        x = x.reshape(BATCH_SIZE, SEQ_LENGTH, NUM_HEADS*HEAD_DIM)
        print(f"after reshapett: {x.shape}")

        x = self.linear(x)
        print(f"after lineartt: {x.shape}")

        return x

In [18]:
cross_attention = CrossAttention()

x_enc = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
x_dec = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)

x = cross_attention(x_enc, x_dec)


x_enc originaltt: torch.Size([1, 200, 512])
x_dec originaltt: torch.Size([1, 200, 512])
nafter kv_lineartt: torch.Size([1, 200, 1024])
after reshapett: torch.Size([1, 200, 8, 128])
after permutett: torch.Size([1, 8, 200, 128])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
nafter q_lineartt: torch.Size([1, 200, 512])
after reshapett: torch.Size([1, 200, 8, 64])
after permute (q)t: torch.Size([1, 8, 200, 64])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
multipliedtt: torch.Size([1, 8, 200, 200])
scaledttt: torch.Size([1, 8, 200, 200])
attn_output_weights: torch.Size([1, 8, 200, 200])
attn_outputtt: torch.Size([1, 8, 200, 64])
nattn_outputtt: torch.Size([1, 8, 200, 64])
attn_output_weightst: torch.Size([1, 8, 200, 200])
after permutett: torch.Size([1, 200, 8, 64])
after reshapett: torch.Size([1, 200, 512])
after lineartt: torch.Size([1, 200, 512])


In [19]:
class FeedForward(nn.Module):

    def __init__(self):
        super().__init__()

        self.linear_0 = nn.Linear(D_MODEL, HIDDEN_DIM)  #(1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=DROP_PROB)  #(2)
        self.linear_1 = nn.Linear(HIDDEN_DIM, D_MODEL)  #(3)

    def forward(self, x):
        print(f"originalt: {x.shape}")

        x = self.linear_0(x)
        print(f"after linear_0t: {x.shape}")

        x = self.relu(x)
        print(f"after relut: {x.shape}")

        x = self.dropout(x)
        print(f"after dropoutt: {x.shape}")

        x = self.linear_1(x)
        print(f"after linear_1t: {x.shape}")

        return x

In [20]:
class LayerNorm(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps  #(1)
        self.gamma = nn.Parameter(torch.ones(D_MODEL), requires_grad=True)  #(2)
        self.beta = nn.Parameter(torch.zeros(D_MODEL), requires_grad=True)  #(3)

    def forward(self, x):  #(4)
        print(f"originalt: {x.shape}")

        mean = x.mean(dim=[-1], keepdim=True)  #(5)
        print(f"meantt: {mean.shape}")

        var = ((x - mean) ** 2).mean(dim=[-1], keepdim=True)  #(6)
        print(f"vartt: {var.shape}")

        stddev = (var + self.eps).sqrt()  #(7)
        print(f"stddevtt: {stddev.shape}")

        x = (x - mean) / stddev  #(8)
        print(f"normalizedt: {x.shape}")

        x = (self.gamma * x) + self.beta  #(9)
        print(f"after scaling and shiftingt: {x.shape}")

        return x

In [21]:
layer_norm = LayerNorm()

x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
x = layer_norm(x)

originalt: torch.Size([1, 200, 512])
meantt: torch.Size([1, 200, 1])
vartt: torch.Size([1, 200, 1])
stddevtt: torch.Size([1, 200, 1])
normalizedt: torch.Size([1, 200, 512])
after scaling and shiftingt: torch.Size([1, 200, 512])


In [22]:
class Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.self_attention = SelfAttention(look_ahead_mask=False)  #(1)
        self.dropout_0 = nn.Dropout(DROP_PROB)  #(2)
        self.layer_norm_0 = LayerNorm()         #(3)
        self.feed_forward = FeedForward()
        self.dropout_1 = nn.Dropout(DROP_PROB)  #(4)
        self.layer_norm_1 = LayerNorm()         #(5)

    def forward(self, x):
        residual = x
        print(f"original &amp; residualt: {x.shape}")

        x = self.self_attention(x)  #(6)
        print(f"after self attentiont: {x.shape}")

        x = self.dropout_0(x)  #(7)
        print(f"after dropouttt: {x.shape}")

        x = self.layer_norm_0(x + residual)  #(8)
        print(f"after layer normt: {x.shape}")

        residual = x
        print(f"nx &amp; residualtt: {x.shape}")

        x = self.feed_forward(x)  #(9)
        print(f"after feed forwardt: {x.shape}")

        x = self.dropout_1(x)
        print(f"after dropouttt: {x.shape}")

        x = self.layer_norm_1(x + residual)
        print(f"after layer normt: {x.shape}")

        return x

In [23]:
encoder = Encoder()

x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
x = encoder(x)

original &amp; residualt: torch.Size([1, 200, 512])
originaltt: torch.Size([1, 200, 512])
after qkv_lineart: torch.Size([1, 200, 1536])
after reshapett: torch.Size([1, 200, 8, 192])
after permutett: torch.Size([1, 8, 200, 192])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
multipliedtt: torch.Size([1, 8, 200, 200])
scaledttt: torch.Size([1, 8, 200, 200])
attn_output_weights: torch.Size([1, 8, 200, 200])
attn_outputtt: torch.Size([1, 8, 200, 64])
attn_outputtt: torch.Size([1, 8, 200, 64])
attn_output_weightst: torch.Size([1, 8, 200, 200])
after permutett: torch.Size([1, 200, 8, 64])
after reshapett: torch.Size([1, 200, 512])
after lineartt: torch.Size([1, 200, 512])
after self attentiont: torch.Size([1, 200, 512])
after dropouttt: torch.Size([1, 200, 512])
originalt: torch.Size([1, 200, 512])
meantt: torch.Size([1, 200, 1])
vartt: 

In [24]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.self_attention = SelfAttention(look_ahead_mask=True)  #(1)
        self.dropout_0 = nn.Dropout(DROP_PROB)  #(2)
        self.layer_norm_0 = LayerNorm()

        self.cross_attention = CrossAttention()  #(3)
        self.dropout_1 = nn.Dropout(DROP_PROB)  #(4)
        self.layer_norm_1 = LayerNorm()

        self.feed_forward = FeedForward()
        self.dropout_2 = nn.Dropout(DROP_PROB)  #(5)
        self.layer_norm_2 = LayerNorm()

    def forward(self, x_enc, x_dec):  #(6)
        residual = x_dec
        print(f"x_dec &amp; residualt: {x_dec.shape}")

        x_dec = self.self_attention(x_dec)  #(7)
        print(f"after self attentiont: {x_dec.shape}")

        x_dec = self.dropout_0(x_dec)
        print(f"after dropouttt: {x_dec.shape}")

        x_dec = self.layer_norm_0(x_dec + residual)  #(8)
        print(f"after layer normt: {x_dec.shape}")

        residual = x_dec
        print(f"nx_dec &amp; residualt: {x_dec.shape}")

        x_dec = self.cross_attention(x_enc, x_dec)  #(9)
        print(f"after cross attentiont: {x_dec.shape}")

        x_dec = self.dropout_1(x_dec)
        print(f"after dropouttt: {x_dec.shape}")

        x_dec = self.layer_norm_1(x_dec + residual)
        print(f"after layer normt: {x_dec.shape}")

        residual = x_dec
        print(f"nx_dec &amp; residualt: {x_dec.shape}")

        x_dec = self.feed_forward(x_dec)  #(10)
        print(f"after feed forwardt: {x_dec.shape}")

        x_dec = self.dropout_2(x_dec)
        print(f"after dropouttt: {x_dec.shape}")

        x_dec = self.layer_norm_2(x_dec + residual)
        print(f"after layer normt: {x_dec.shape}")

        return x_dec

In [25]:
decoder = Decoder()

x_enc = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)
x_dec = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_MODEL)

x = decoder(x_enc, x_dec)

x_dec &amp; residualt: torch.Size([1, 200, 512])
originaltt: torch.Size([1, 200, 512])
after qkv_lineart: torch.Size([1, 200, 1536])
after reshapett: torch.Size([1, 200, 8, 192])
after permutett: torch.Size([1, 8, 200, 192])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
multipliedtt: torch.Size([1, 8, 200, 200])
scaledttt: torch.Size([1, 8, 200, 200])
maskttt: torch.Size([200, 200])
attn_output_weights: torch.Size([1, 8, 200, 200])
attn_outputtt: torch.Size([1, 8, 200, 64])
attn_outputtt: torch.Size([1, 8, 200, 64])
attn_output_weightst: torch.Size([1, 8, 200, 200])
after permutett: torch.Size([1, 200, 8, 64])
after reshapett: torch.Size([1, 200, 512])
after lineartt: torch.Size([1, 200, 512])
after self attentiont: torch.Size([1, 200, 512])
after dropouttt: torch.Size([1, 200, 512])
originalt: torch.Size([1, 200, 512])
meantt: to

In [27]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_embedding = InputEmbedding()  #(1)
        self.output_embedding = OutputEmbedding()  #(2)

        self.positional_encoding = PositionalEncoding()  #(3)

        self.encoders = nn.ModuleList([Encoder() for _ in range(N)])  #(4)
        self.decoders = nn.ModuleList([Decoder() for _ in range(N)])  #(5)

        self.linear = nn.Linear(D_MODEL, VOCAB_SIZE_DST)  #(6)

    def forward(self, x_enc_raw, x_dec_raw):  #(1)
        print(f"x_enc_rawtt: {x_enc_raw.shape}")
        print(f"x_dec_rawtt: {x_dec_raw.shape}")

        # Encoder
        x_enc = self.input_embedding(x_enc_raw)  #(2)
        print(f"nafter input embeddingt: {x_enc.shape}")

        x_enc = x_enc + self.positional_encoding()  #(3)
        print(f"after pos encodingt: {x_enc.shape}")

        for i, encoder in enumerate(self.encoders):
            x_enc = encoder(x_enc)  #(4)
            print(f"after encoder #{i}t: {x_enc.shape}")

        # Decoder
        x_dec = self.output_embedding(x_dec_raw)  #(5)
        print(f"nafter output embeddingt: {x_dec.shape}")

        x_dec = x_dec + self.positional_encoding()  #(6)
        print(f"after pos encodingt: {x_dec.shape}")

        for i, decoder in enumerate(self.decoders):
            x_dec = decoder(x_enc, x_dec)  #(7)
            print(f"after decoder #{i}t: {x_dec.shape}")

        x = self.linear(x_dec)  #(8)
        print(f"nafter lineartt: {x.shape}")

        return x


In [28]:
transformer = Transformer()

x_enc_raw = torch.randint(0, VOCAB_SIZE_SRC, (BATCH_SIZE, SEQ_LENGTH))
x_dec_raw = torch.randint(0, VOCAB_SIZE_DST, (BATCH_SIZE, SEQ_LENGTH))

y = transformer(x_enc_raw, x_dec_raw).shape

x_enc_rawtt: torch.Size([1, 200])
x_dec_rawtt: torch.Size([1, 200])
original: torch.Size([1, 200])
after embedding: torch.Size([1, 200, 512])
nafter input embeddingt: torch.Size([1, 200, 512])
postt: torch.Size([200, 1])
denominatort: torch.Size([256])
even_pos_embed: torch.Size([200, 256])
stackedtt: torch.Size([200, 256, 2])
pos_embedt: torch.Size([200, 512])
after pos encodingt: torch.Size([1, 200, 512])
original &amp; residualt: torch.Size([1, 200, 512])
originaltt: torch.Size([1, 200, 512])
after qkv_lineart: torch.Size([1, 200, 1536])
after reshapett: torch.Size([1, 200, 8, 192])
after permutett: torch.Size([1, 8, 200, 192])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
multipliedtt: torch.Size([1, 8, 200, 200])
scaledttt: torch.Size([1, 8, 200, 200])
attn_output_weights: torch.Size([1, 8, 200, 200])
attn_outputtt: torch.Siz

In [29]:
transformer = Transformer()
summary(transformer, input_data=(x_enc_raw, x_dec_raw))

x_enc_rawtt: torch.Size([1, 200])
x_dec_rawtt: torch.Size([1, 200])
original: torch.Size([1, 200])
after embedding: torch.Size([1, 200, 512])
nafter input embeddingt: torch.Size([1, 200, 512])
postt: torch.Size([200, 1])
denominatort: torch.Size([256])
even_pos_embed: torch.Size([200, 256])
stackedtt: torch.Size([200, 256, 2])
pos_embedt: torch.Size([200, 512])
after pos encodingt: torch.Size([1, 200, 512])
original &amp; residualt: torch.Size([1, 200, 512])
originaltt: torch.Size([1, 200, 512])
after qkv_lineart: torch.Size([1, 200, 1536])
after reshapett: torch.Size([1, 200, 8, 192])
after permutett: torch.Size([1, 8, 200, 192])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
qttt: torch.Size([1, 8, 200, 64])
kttt: torch.Size([1, 8, 200, 64])
vttt: torch.Size([1, 8, 200, 64])
multipliedtt: torch.Size([1, 8, 200, 200])
scaledttt: torch.Size([1, 8, 200, 200])
attn_output_weights: torch.Size([1, 8, 200, 200])
attn_outputtt: torch.Siz

Layer (type:depth-idx)                   Output Shape              Param #
Transformer                              [1, 200, 120]             --
├─InputEmbedding: 1-1                    [1, 200, 512]             --
│    └─Embedding: 2-1                    [1, 200, 512]             51,200
├─PositionalEncoding: 1-2                [200, 512]                --
├─ModuleList: 1-3                        --                        --
│    └─Encoder: 2-2                      [1, 200, 512]             --
│    │    └─SelfAttention: 3-1           [1, 200, 512]             1,050,624
│    │    └─Dropout: 3-2                 [1, 200, 512]             --
│    │    └─LayerNorm: 3-3               [1, 200, 512]             1,024
│    │    └─FeedForward: 3-4             [1, 200, 512]             2,099,712
│    │    └─Dropout: 3-5                 [1, 200, 512]             --
│    │    └─LayerNorm: 3-6               [1, 200, 512]             1,024
│    └─Encoder: 2-3                      [1, 200, 512]       